# Neural Networks Day 1: From Gradient Descent to Backpropagation

## Learning Objectives

By the end of this lab, you will be able to:

1. **Understand Gradient Descent**: Visualize and implement gradient descent optimization on a 2D surface
2. **Analyze Hyperparameters**: Observe how learning rate and initialization affect convergence
3. **Implement a Neural Network from Scratch**: Build a complete forward and backward pass using only NumPy
4. **Master Backpropagation**: Understand how errors flow backward through a network to update weights
5. **Solve the XOR Problem**: Train a multi-layer network to learn a non-linear classification task

---

## Setup: Import Libraries

We'll use NumPy for numerical computations, Matplotlib for static plots, and Plotly for interactive 3D visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

---

# Part 1: Gradient Descent Visualization

## Understanding the Optimization Landscape

Gradient descent is the fundamental algorithm behind training neural networks. To understand it intuitively, we'll visualize how it navigates a complex 2D surface.

### The Target Function

We'll minimize a function from Simone Scardapane's book *"Alice's Adventures in a Differentiable Wonderland"*:

$$f(x_1, x_2) = \sin(x_1)\cos(x_2) + \sin(0.5x_1)\cos(0.5x_2)$$

This function has multiple local minima and maxima, making it an excellent test case for optimization algorithms.

In [ ]:
def fct_to_minimize(X: np.array) -> np.array:
    """
    Function taken from Simone Scardapane's book
    "Alice's Adventures in a differentiable wonderland"
    f(X) = sin(x1) cos(x2) + sin(0.5*x1) cos(0.5*x2)
    Illustrated here for an array X of shape (n, 2).
    """
    x1, x2 = X[:, 0], X[:, 1]
    return np.sin(x1) * np.cos(x2) + np.sin(0.5 * x1) * np.cos(0.5 * x2)

## Computing Gradients

The gradient of a function points in the direction of steepest ascent. For gradient descent, we move in the opposite direction.

For our function, the partial derivatives are:

$$\frac{\partial f}{\partial x_1} = \cos(x_1)\cos(x_2) + 0.5\cos(0.5x_1)\cos(0.5x_2)$$

$$\frac{\partial f}{\partial x_2} = -\sin(x_1)\sin(x_2) - 0.5\sin(0.5x_1)\sin(0.5x_2)$$

In [ ]:
def compute_grad(X: np.array) -> np.array:
    # grad = [df/dx1, df/dx2]
    x1, x2 = X[:, 0], X[:, 1]

    df_dx1 = np.cos(x1) * np.cos(x2) + 0.5 * np.cos(0.5 * x1) * np.cos(0.5 * x2)
    df_dx2 = -np.sin(x1) * np.sin(x2) - 0.5 * np.sin(0.5 * x1) * np.sin(0.5 * x2)
    return np.stack([df_dx1, df_dx2]).squeeze(-1)

## The Gradient Descent Algorithm

Gradient descent updates parameters iteratively:

$$X_{new} = X_{old} - \eta \cdot \nabla f(X_{old})$$

Where:
- $\eta$ (learning rate): Controls step size
- $\nabla f$: Gradient of the function

**Key Insight**: The learning rate is crucial—too small and convergence is slow; too large and we might overshoot or diverge.

In [ ]:
def gradient_descent(X: np.array, lr: float, n_iter: int) -> np.array:
    """
    Perform n_iter steps of gradient descent with learning rate lr
    on the function fct_to_minimize starting at point X.
    """
    traj = []
    X = X.copy()
    # print(X.shape)
    for i in range(n_iter):
        grad = compute_grad(X)
        X -= lr * grad
        traj.append(X.copy())
    return X, np.array(traj)

## Interactive 3D Visualization

This function creates an interactive 3D plot of our function surface, optionally showing the trajectory of gradient descent from start (red) to end (green).

In [ ]:
def plot_function_plotly(traj: np.array = None):
    x1 = np.linspace(0, 10, 100)
    x2 = np.linspace(0, 10, 100)
    X1, X2 = np.meshgrid(x1, x2)
    X_flat = np.stack([X1.ravel(), X2.ravel()], axis=1)
    Z = fct_to_minimize(X_flat).reshape(100, 100)

    # Create figure
    fig = go.Figure()

    # Add 3D surface plot
    fig.add_trace(go.Surface(
        x=X1, y=X2, z=Z,
        colorscale="rdylbu_r",
        opacity=0.7
    ))

    # Plot trajectory if provided
    if traj is not None:
        traj = np.array(traj)
        traj = traj.squeeze(1)  # Remove singleton dimension
        x_traj, y_traj = traj[:, 0], traj[:, 1]
        z_traj = fct_to_minimize(traj)

        # Add trajectory as a line
        fig.add_trace(go.Scatter3d(
            x=x_traj, y=y_traj, z=z_traj,
            mode="lines+markers",
            marker=dict(size=3, color='black'),
            line=dict(color="black", width=3),
            name=""
        ))

        # Start and end points
        fig.add_trace(go.Scatter3d(
            x=[x_traj[0]], y=[y_traj[0]], z=[z_traj[0]],
            mode="markers",
            marker=dict(size=6, color='red'),
            name="Start"
        ))

        fig.add_trace(go.Scatter3d(
            x=[x_traj[-1]], y=[y_traj[-1]], z=[z_traj[-1]],
            mode="markers",
            marker=dict(size=6, color='green'),
            name="End"
        ))

    # Customize layout
    fig.update_layout(
        title="",
        scene=dict(
            xaxis_title="x1",
            yaxis_title="x2",
            zaxis_title="f(x1, x2)"
        ),
        showlegend=False
    )

    fig.show()

## Experiment 1: Effect of Random Initialization

**Question**: How does the starting point affect which minimum we converge to?

Run this cell multiple times to see how different random starting points lead to different trajectories and potentially different local minima.

**Observation Tips**:
- Notice how the trajectory follows the steepest descent path
- Different starting points may converge to different valleys
- The surface has multiple local minima—gradient descent finds the nearest one

In [ ]:
# How does selecting different starting points affect the trajectory?
# Play with this cell around - call it multiple times! We don't have here a deterministic output, so every time the starting point will differ!

for i in range(5):
    X_0 = np.random.uniform(0, 10, (1, 2))
    lr = 0.2
    n_iter = 100
    X_min, traj = gradient_descent(X_0, lr, n_iter)
    plot_function_plotly(traj=traj)

## Experiment 2: Effect of Number of Iterations

**Question**: How does the number of iterations affect convergence?

Here we use a fixed starting point and gradually increase the number of gradient steps from 1 to 10.

**Observation Tips**:
- Watch how the trajectory extends with more iterations
- Notice when the path starts to flatten (convergence)
- Early steps are large; later steps become smaller as the gradient decreases

In [ ]:
# How does the number of iterations affect the trajectory?
# Here we selected a nice starting point X_0 where the gradient descent is easy to track.

X_0 = np.array([[8, 7]]).astype(float)
lr = 0.2
n_iter = np.arange(1, 11, 1)

# we iterate from 1 to 10 steps in total
for n in n_iter:
    X_min, traj = gradient_descent(X_0, lr, n)
    plot_function_plotly(traj=traj)

## Experiment 3: Effect of Learning Rate

**Question**: How does the learning rate affect optimization?

We test four different learning rates:
- **0.002**: Very small—slow convergence
- **0.2**: Moderate—good balance
- **1.0**: Large—may oscillate
- **2.5**: Very large—may diverge

**Observation Tips**:
- Small LR: Smooth but slow progress
- Large LR: Fast initial progress but may overshoot
- Too large LR: The algorithm may fail to converge or even diverge

In [ ]:
# How does learning rate affect the trajectory?
X_0 = np.array([[8, 7]]).astype(float)
lr = np.array([0.002, 0.2, 1.0, 2.5])

for l in lr:
    n_iter = 20
    X_min, traj = gradient_descent(X_0, l, n_iter)
    plot_function_plotly(traj=traj)

---

# Part 2: Building a Neural Network from Scratch

## The XOR Problem

Now we'll implement a complete neural network using only NumPy. Our goal is to solve the **XOR problem**—a classic task that requires a hidden layer (cannot be solved by a single linear classifier).

### Network Architecture

```
Input Layer (2 neurons + bias) → Hidden Layer (2 neurons + bias) → Output Layer (1 neuron)
```

### Activation Functions

- **Hidden Layer**: Hyperbolic Tangent (tanh)
  - $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
  - Output range: (-1, 1)
  - Derivative: $1 - \tanh^2(x)$

- **Output Layer**: Logistic Sigmoid
  - $\sigma(x) = \frac{1}{1 + e^{-x}}$
  - Output range: (0, 1)
  - Derivative: $\sigma(x)(1 - \sigma(x))$

### Backpropagation Overview

Backpropagation computes gradients by applying the **chain rule** from the output back to the inputs:

1. **Forward Pass**: Compute predictions through the network
2. **Compute Error**: Compare predictions to targets
3. **Backward Pass**: Propagate error gradients backward
4. **Weight Update**: Adjust weights using gradient descent

In [ ]:
def neuron_w(input_count):
  weights = np.zeros(input_count+1)
  for i in range(1, (input_count+1)):
    weights[i] = np.random.uniform(-1.0, 1.0)
  return weights

def show_learning():
  print('Current weights:')
  for i, w in enumerate(n_w):
    print('neuron ', i,
          ': w0 =', '%5.2f' % w[0],
          ', w1 =', '%5.2f' % w[1], ', w2 =',
          '%5.2f' % w[2])
  print('----------------')

def forward_pass(x):
  global n_y
  n_y[0] = np.tanh(np.dot(n_w[0], x)) # Neuron 0
  n_y[1] = np.tanh(np.dot(n_w[1], x)) # Neuron 1
  n2_inputs = np.array([1.0, n_y[0], n_y[1]]) # 1.0 is bias
  z2 = np.dot(n_w[2], n2_inputs)
  n_y[2] = 1.0 / (1.0 + np.exp(-z2))

def backward_pass(y_truth):
  global n_error
  error_prime = -(y_truth - n_y[2]) # Derivative of loss-func
  derivative = n_y[2] * (1.0 - n_y[2]) # Logistic derivative
  n_error[2] = error_prime * derivative
  derivative = 1.0 - n_y[0]**2 # tanh derivative
  n_error[0] = n_w[2][1] * n_error[2] * derivative
  derivative = 1.0 - n_y[1]**2 # tanh derivative
  n_error[1] = n_w[2][2] * n_error[2] * derivative

def adjust_weights(x):
  global n_w
  n_w[0] -= (x * LEARNING_RATE * n_error[0])
  n_w[1] -= (x * LEARNING_RATE * n_error[1])
  n2_inputs = np.array([1.0, n_y[0], n_y[1]]) # 1.0 is bias
  n_w[2] -= (n2_inputs * LEARNING_RATE * n_error[2])

## Training the Network

### XOR Truth Table

| x1 | x2 | XOR (y) |
|:---:|:---:|:---:|:---:|
| -1 | -1 | 0 |
| -1 | 1 | 1 |
| 1 | -1 | 1 |
| 1 | 1 | 0 |

The training loop:
1. Randomly shuffle the order of training examples
2. For each example: forward → backward → weight update
3. Check if all predictions are correct (convergence)
4. Repeat until converged

**Note**: We use stochastic gradient descent (SGD) with one example at a time.

In [ ]:
np.random.seed(3) # To make repeatable
index_list = [0, 1, 2, 3] # Used to randomize order
LEARNING_RATE = 0.1

# Examples - input features
x_train = [np.array([1.0, -1.0, -1.0]),
           np.array([1.0, -1.0, 1.0]),
           np.array([1.0, 1.0, -1.0]),
           np.array([1.0, 1.0, 1.0])]

# Examples - output (ground truth)
y_train = [0.0, 1.0, 1.0, 0.0]

# Network initialization
n_w = [neuron_w(2), neuron_w(2), neuron_w(2)]
n_y = [0, 0, 0]
n_error = [0, 0, 0]
# Network training loop
all_correct = False
while not all_correct: # Train until converged
  all_correct = True

  np.random.shuffle(index_list) # Randomize order
  for i in index_list: # Train on all examples
    forward_pass(x_train[i])
    backward_pass(y_train[i])
    adjust_weights(x_train[i])
    show_learning() # Show updated weights
  for i in range(len(x_train)): # Check if converged
    forward_pass(x_train[i])
    print('x1 =', '%4.1f' % x_train[i][1], ', x2 =',
          '%4.1f' % x_train[i][2], ', y =',
          '%.4f' % n_y[2])
    if (((y_train[i] < 0.5) and (n_y[2] >= 0.5))
    or ((y_train[i] >= 0.5) and (n_y[2] < 0.5))):
        all_correct = False

## Training with Fixed Epochs and MSE Tracking

This modified version trains for a fixed number of epochs (1000) and tracks the Mean Squared Error (MSE) over time. This allows us to visualize how the error decreases during training.

**Key Differences**:
- Fixed number of epochs instead of convergence check
- MSE computation for monitoring training progress
- Visualization of error curve

In [ ]:
# CHANGING THINGS FOR QUESTION 11
np.random.seed(3) # To make repeatable
index_list = [0, 1, 2, 3] # Used to randomize order
LEARNING_RATE = 0.1

# Examples - input features
x_train = [np.array([1.0, -1.0, -1.0]),
           np.array([1.0, -1.0, 1.0]),
           np.array([1.0, 1.0, -1.0]),
           np.array([1.0, 1.0, 1.0])]

# Examples - output (ground truth)
y_train = [0.0, 1.0, 1.0, 0.0]

# Network
EPOCH = 0
MSE = []
n_w = [neuron_w(2), neuron_w(2), neuron_w(2)]
n_y = [0, 0, 0]
n_error = [0, 0, 0]
# Network training loop
while EPOCH < 1000: # Train for 1000 EPOCHS
  EPOCH += 1

  np.random.shuffle(index_list) # Randomize order
  for i in index_list: # Train on all examples
    forward_pass(x_train[i])
    backward_pass(y_train[i])
    adjust_weights(x_train[i])
    show_learning() # Show updated weights
    print('EPOCH: ', EPOCH)

  for i in range(len(x_train)): # Check if converged
    forward_pass(x_train[i])
  MSE.append(0.25 * (y_train[i] - n_y[2])**2)

# PLOTTING MSE
epochs = np.arange(1, 1001, 1)

# Create a line plot
fig = go.Figure()

fig.add_trace(go.Scatter(x=epochs, y=MSE, mode='markers', name='Line Plot'))

# Show the figure
fig.show()

---

# Summary

## Key Takeaways

### Gradient Descent
- **Initialization matters**: Different starting points can lead to different local minima
- **Learning rate is crucial**: Too small = slow convergence; Too large = divergence
- **Iterations affect accuracy**: More iterations generally lead to better convergence (up to a point)

### Neural Networks
- **Hidden layers enable non-linearity**: XOR requires a hidden layer to solve
- **Backpropagation chains derivatives**: Error flows backward through the chain rule
- **Activation functions introduce non-linearity**: tanh and sigmoid enable complex decision boundaries

### Mathematical Foundations
- Forward pass: $y = \sigma(W_2 \cdot \tanh(W_1 \cdot x))$
- Backward pass: Apply chain rule to compute $\frac{\partial L}{\partial W}$
- Weight update: $W \leftarrow W - \eta \cdot \frac{\partial L}{\partial W}$

## Next Steps

In the next lab, we'll explore:
- Deep networks with multiple hidden layers
- Modern activation functions (ReLU)
- PyTorch implementation of these concepts